In [6]:
pip install ollama 


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import gradio as gr
import ollama
import pdfplumber
import pandas as pd


# =========================
# PDF TEXT EXTRACT FUNCTION
# =========================
def extract_text_from_pdf(pdf_file):

    text = ""

    with pdfplumber.open(pdf_file) as pdf:

        for page in pdf.pages:

            page_text = page.extract_text()

            if page_text:
                text += page_text + "\n"

    return text


# =========================
# AI ANALYSIS FUNCTION
# =========================
def analyze_resume(job_description, resume_file):

    try:

        # Extract text from PDF
        resume_text = extract_text_from_pdf(resume_file.name)

        # AI Prompt
        prompt = f"""
You are an HR recruiter.

Analyze the following resume.

JOB DESCRIPTION:
{job_description}

RESUME:
{resume_text}

Give:

1. Candidate Name
2. Match Score out of 100
3. Skills Found
4. Missing Skills
5. Strengths
6. Weaknesses
7. Hiring Recommendation
"""

        # AI Response
        response = ollama.chat(
            model='qwen2.5',
            messages=[
                {
                    'role': 'user',
                    'content': prompt
                }
            ]
        )

        result = response['message']['content']

        return result

    except Exception as e:

        return f"❌ Error processing resume:\n\n{str(e)}"


# =========================
# GRADIO UI
# =========================
with gr.Blocks() as demo:

    gr.Markdown("# 🤖 HR Resume Screening AI")

    job_description = gr.Textbox(
        label="Job Description",
        lines=8
    )

    resume_file = gr.File(
        label="Upload Resume PDF",
        file_types=['.pdf']
    )

    analyze_btn = gr.Button("Analyze Resume")

    output = gr.Textbox(
        label="AI Analysis",
        lines=20
    )

    analyze_btn.click(
        fn=analyze_resume,
        inputs=[job_description, resume_file],
        outputs=output
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.
